# Task 3: Generate YAML from the descriptions with 3 Ollama models (Kaggle)

**Before running:** Settings (right panel) → **Accelerator: GPU T4 x2** and **Internet: On**.
Kaggle turns on internet only after phone verification.
Then click **Run All**. Expect about 15–25 minutes in total, most of it downloading the models.

Output files are in `/kaggle/working`, under the **Output** tab on the right:
- `generated_yaml_for_sheet.csv`: one row per project, one column per model, ready to paste into the Google Sheet
- `generated_yaml_long.csv`: one row per project × model, with generation time
- `outputs/<model>/<no>.yml`: each generated YAML as its own file

## 1. Install Ollama and start the server

In [ ]:
!apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!pip -q install pyyaml

import subprocess, time, requests, os
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
server = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434", timeout=2); print("Ollama server is up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start - see /kaggle/working/ollama.log")

## 2. Pull the 3 models

In [ ]:
MODELS = ["qwen2.5-coder:7b", "qwen2.5-coder:14b", "deepseek-coder-v2:16b"]
for m in MODELS:
    print("pulling", m, "...")
    r = subprocess.run(["ollama", "pull", m], capture_output=True, text=True)
    print("  ok" if r.returncode == 0 else r.stderr[-500:])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## 3. The 5 descriptions (from the Google Sheet)

In [ ]:
ITEMS = [
  {
    "no": 1,
    "title": "Azure Pipeline Basic (Python + pytest)",
    "repo": "https://github.com/atulkamble/azure-pipeline-basic",
    "description": "Create an Azure Pipelines YAML file (azure-pipelines.yml) for a Python project. The pipeline is triggered by commits to the main branch and runs on a Microsoft-hosted ubuntu-latest agent. Define a pipeline variable named pythonVersion with the value '3.11'. The pipeline has three steps: (1) the UsePythonVersion@0 task, selecting the Python version from the pythonVersion variable; (2) a script step with display name 'Install dependencies' that upgrades pip and installs the packages listed in requirements.txt; (3) a script step with display name 'Run tests' that runs pytest in quiet mode (pytest -q)."
  },
  {
    "no": 2,
    "title": "JavaScript E2E Express Server (Node.js + JUnit test results)",
    "repo": "https://github.com/Azure-Samples/js-e2e-express-server",
    "description": "Create an Azure Pipelines YAML file for a Node.js Express server project. The pipeline is triggered by commits to the main branch and runs on a Microsoft-hosted ubuntu-latest agent. It has three steps: (1) the NodeTool@0 task installing Node.js version '10.x', with display name 'Install Node.js'; (2) a script step (no display name) that runs npm install, then npm run build, then npm test; (3) the PublishTestResults@2 task that publishes test results in JUnit format from files matching '**/TEST-*.xml'. This publish step has the display name 'npm install and build'."
  },
  {
    "no": 3,
    "title": "Azure Batch Samples (Python, Java, .NET)",
    "repo": "https://github.com/Azure-Samples/azure-batch-samples",
    "description": "Create an Azure Pipelines YAML file that tests the Azure Batch sample code in three languages. The pipeline is triggered by commits to the master branch. Define three independent jobs, each running on a Microsoft-hosted windows-2019 agent. Job 'PythonTest' (display name 'Run lint tests') uses a matrix strategy with two configurations: PythonMinVersion with python.version '3.7' and PythonMaxVersion with python.version '3.10'. Its steps are: the UsePythonVersion@0 task with versionSpec '$(python.version)', then one script that upgrades setuptools, installs requirements from $(System.DefaultWorkingDirectory)/Python/Batch/requirements.txt, installs coveralls, flake8, pytest, pytest-cov and requests_mock, and runs flake8 on $(System.DefaultWorkingDirectory)/Python/Batch. Job 'JavaTest' has one Maven@3 task using the POM file $(System.DefaultWorkingDirectory)/Java/pom.xml, mavenOptions '-Xmx3072m', javaHomeOption 'JDKVersion', jdkVersionOption '1.8', jdkArchitectureOption 'x64', publishJUnitResults false, testResultsFiles '**/TEST-*.xml' and goals 'package'. Job 'DotNetTest' has two DotNetCoreCLI@2 tasks: the first runs the 'restore' command on '**/*.csproj' with feedsToUse 'select', the second runs the 'build' command on '**/*.csproj'."
  },
  {
    "no": 4,
    "title": "Todo App: Node.js + MongoDB on Azure (azd deploy)",
    "repo": "https://github.com/Azure-Samples/todo-nodejs-mongo",
    "description": "Create a GitHub Actions workflow that deploys a Node.js + MongoDB todo application to Azure using the Azure Developer CLI (azd). The workflow runs on manual dispatch (workflow_dispatch) and on pushes to the main or master branch. Grant the permissions id-token: write and contents: read so it can log in with OpenID Connect federated credentials. It has one job named 'build' on ubuntu-latest with job-level environment variables AZURE_CLIENT_ID, AZURE_TENANT_ID and AZURE_SUBSCRIPTION_ID taken from repository variables (vars) and AZURE_CREDENTIALS taken from secrets. The steps are: 'Checkout' using actions/checkout@v4; 'Install azd' using Azure/setup-azd@v2; 'Install Nodejs' using actions/setup-node@v4 with node-version 18; 'Log in with Azure (Federated Credentials)', which runs only if AZURE_CLIENT_ID is not empty and runs azd auth login with the client ID, --federated-credential-provider \"github\" and the tenant ID in a pwsh shell; 'Log in with Azure (Client Credentials)', which runs only if AZURE_CREDENTIALS is not empty, parses the AZURE_CREDENTIALS JSON in pwsh, masks the client secret, and runs azd auth login with the client ID, client secret and tenant ID; 'Provision Infrastructure', which runs azd provision --no-prompt with environment variables AZURE_ENV_NAME, AZURE_LOCATION and AZURE_SUBSCRIPTION_ID from vars and AZD_INITIAL_ENVIRONMENT_CONFIG from secrets; 'Deploy Application', which runs azd deploy --no-prompt with AZURE_ENV_NAME, AZURE_LOCATION and AZURE_SUBSCRIPTION_ID from vars."
  },
  {
    "no": 5,
    "title": "Java EE 7 Petstore: Deploy to Azure Web App",
    "repo": "https://github.com/Azure-Samples/Java-application-petstore-ee7",
    "description": "Create a GitHub Actions workflow named 'Deploy Java Web App to Azure' that builds a Java EE 7 petstore application with Maven and deploys the WAR file to Azure App Service. It runs on push and on pull_request. Define workflow-level environment variables: AZURE_WEBAPP_NAME = JavaPetstore; AZURE_WEBAPP_PACKAGE_PATH = ${{ github.workspace }}; JAVA_VERSION = '1.8'; AZURE_WEBAPP_PUBLISH_PROFILE from the secret AZURE_WEBAPP_PUBLISH_PROFILE; CONTAINER_REGISTRY = ${{ secrets.REGISTRY_USERNAME }}.azurecr.io; AZURE_RESOURCE_GROUP = ActionsDemo; AZURE_APP_PLAN = ActionPlan. It has one job, 'build-and-deploy', on ubuntu-latest with these steps: check out the code with actions/checkout@master; 'Set up JDK 1.8' using actions/setup-java@v1 with java-version from JAVA_VERSION; 'Build with Maven' running mvn clean install --file pom.xml -Dmaven.test.skip=true; 'Azure authentication' using azure/login@v1 with creds from the secret AZURE_CREDENTIALS; 'Deploy to Azure WebApp' using azure/webapps-deploy@v2 with app-name from AZURE_WEBAPP_NAME and package '${{ env.AZURE_WEBAPP_PACKAGE_PATH }}/target/*.war'."
  }
]
for it in ITEMS:
    print(f"{it['no']}. {it['title']}  ({len(it['description'].split())} words)")

## 4. Generate YAML
Every model gets the same prompt, with `temperature = 0` and a fixed seed, so you get the same output if you run it again.
The prompt is only an instruction plus the description. Nothing from the original YAML is given to the model.

In [ ]:
import re, json, yaml

SYSTEM = ("You are an experienced DevOps engineer. Write the CI/CD YAML file described by the user. "
          "Follow the description exactly: use the exact names, versions, paths and values it gives, "
          "do not add anything it does not ask for, and do not leave anything out. "
          "Output only the YAML inside a single ```yaml code block, with no explanation.")
OPTIONS = {"temperature": 0, "seed": 42, "num_ctx": 8192, "num_predict": 2048}

def extract_yaml(text):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    blocks = re.findall(r"```(?:ya?ml|YAML)?\s*\n(.*?)```", text, flags=re.S)
    if blocks:
        return max(blocks, key=len).strip()
    m = re.search(r"```(?:ya?ml)?\s*\n(.*)", text, flags=re.S)
    return (m.group(1) if m else text).strip()

def parses(y):
    try:
        yaml.safe_load(y); return "yes"
    except Exception as e:
        return "no: " + str(e).split("\n")[0]

results = []
for model in MODELS:
    folder = f"/kaggle/working/outputs/{model.replace(':', '_')}"
    os.makedirs(folder, exist_ok=True)
    for it in ITEMS:
        t = time.time()
        r = requests.post("http://127.0.0.1:11434/api/generate", json={
            "model": model, "system": SYSTEM, "prompt": it["description"],
            "stream": False, "options": OPTIONS}, timeout=3600)
        r.raise_for_status()
        raw = r.json()["response"]
        y = extract_yaml(raw)
        secs = round(time.time() - t, 1)
        open(f"{folder}/{it['no']}.yml", "w").write(y)
        open(f"{folder}/{it['no']}.raw.txt", "w").write(raw)
        results.append({"no": it["no"], "title": it["title"], "model": model,
                        "valid_yaml": parses(y), "seconds": secs, "generated_yaml": y})
        print(f"{model:24s} #{it['no']}  {secs:6.1f}s  valid YAML: {parses(y)[:3]}")
    # unload the model to free GPU memory before the next one
    requests.post("http://127.0.0.1:11434/api/generate", json={"model": model, "keep_alive": 0})

## 5. Save for the Google Sheet

In [ ]:
import pandas as pd
long = pd.DataFrame(results)
long.to_csv("/kaggle/working/generated_yaml_long.csv", index=False)

wide = long.pivot(index=["no", "title"], columns="model", values="generated_yaml").reset_index()
wide = wide[["no", "title"] + MODELS]
wide.columns = ["No", "Title"] + [f"Generated YAML ({m})" for m in MODELS]
wide.to_csv("/kaggle/working/generated_yaml_for_sheet.csv", index=False)

display(long[["no", "title", "model", "valid_yaml", "seconds"]])
print("Saved: generated_yaml_for_sheet.csv and generated_yaml_long.csv (Output tab)")

## 6. Print every generated YAML (to copy into the sheet)

In [ ]:
for it in ITEMS:
    print("=" * 100)
    print(f"#{it['no']}  {it['title']}")
    for row in results:
        if row["no"] == it["no"]:
            print("-" * 100)
            print(f"[{row['model']}]  valid YAML: {row['valid_yaml']}   ({row['seconds']}s)\n")
            print(row["generated_yaml"])